In [ ]:
import requests
import csv
import datetime
import time
import pandas as pd

In [ ]:

team_codes = {
    1 : "NJD", 2 : "NYI", 3 : "NYR",
    4 : "PHI", 5 : "PIT", 6 : "BOS",
    7 : "BUF", 8 : "MTL", 9 : "OTT",
    10 : "TOR", 12 : "CAR", 13 : "FLA",
    14 : "TBL", 15 : "WSH", 16 : "CHI",
    17 : "DET", 18 : "NSH", 19 : "STL",
    20 : "CGY", 21 : "COL", 22 : "EDM",
    23 : "VAN", 24 : "ANA", 25 : "DAL",
    26 : "LAK", 27 : "PHX", 28 : "SJS",
    29 : "CBJ", 30 : "MIN", 52 : "WPG",
    53 : "ARI", 54 : "VGK", 55 : "SEA",
    59 : "UTA"

}

def time_to_float(time_str):
    minutes, seconds = map(int, time_str.split(":"))
    return minutes + seconds / 60

def get_player_rating(player):
    xPoints = (player['goals'] + player['assists']) / player['gamesPlayed']
    xBlocks = player['blockedShots'] / player['gamesPlayed']
    xToi = player['icetime'] / player['gamesPlayed']
    xHits = player['hits'] / player['gamesPlayed']
    xTkwy = player['takeaways'] / player['gamesPlayed']
    xGvwy = player['giveaways'] / player['gamesPlayed']
    # temporary: avg faceoff needs to take into acc num faceoffs total which is in play by play data not boxscore
    xFO = player['AccFaceOffPercentage'] / player['gamesPlayed']
    xPim = player['pim'] / player['gamesPlayed']
    xPlusMinus = player['pm'] / player['gamesPlayed']

    if player.loc['position'] == 'D':
        return ( (.3 * xPoints) + (.2 * xBlocks) + (.2 * xToi) + (.1 * xHits) + (.2 * xTkwy) - (.5 * xGvwy) - (.2 * xPim) + (.1 * xPlusMinus) )
    else:
        return ( (.4 * xPoints) + (.05 * xBlocks) + (.2 * xToi) + (.05 * xHits) + (.2 * xTkwy) - (.5 * xGvwy) - (.2 * xPim) + (.1 * xPlusMinus) + (.1 * (xFO - .5)) )

def get_all_player_rating(game_data, year):
    player_ids = []
    for player in game_data["skaters"]:
        player_ids.append(player["playerId"])

    players = pd.read_csv(f'{year}_{year+1}_All_Players_Data.csv')

    player_rating = 0
    for playerId in player_ids:
        index = players.loc[players['playerId'] == playerId].index
        if not index.empty:
            player_rating += get_player_rating(players.loc[index[0]])
    return player_rating

def get_goalie_rating(game_data, away, year):
     goalies = pd.read_csv(f'{year}_{year+1}_All_Goalies_Data_TEST.csv')
     goalie_count = 0
     total_time
     unnormalized_ratings = []
     rating = 0
     for goalie in game_data["goalies"]:
        goalie_count += 1
        index = goalies.loc[goalies['playerId'] == goalie["playerId"]].index

        if not index.empty:
            goalie_row = goalies.loc[index[0]]  # Extract first matching row
            rating += ( (.005 * goalie_row['gamesPlayed']) + (goalie_row['savePercentage']) - (.2 * goalie_row['GAA']) - (.3 * goalie_row['reboundPercentage']) )


     return rating / goalie_count